In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

from matplotlib import pyplot as plt

In [ ]:
seed = 42
y_col = "Class"

## 1. データの読み込み

In [ ]:
# # 初回のみ実行
# from ucimlrepo import fetch_ucirepo 
  
# dry_bean = fetch_ucirepo(id=602) 
# df = dry_bean.data.original.copy()
# df.to_csv("./data/data.csv", index=False)

In [ ]:
if "df_data" not in locals():
    df_data = pd.read_csv("./data/data.csv")

In [ ]:
df_data.head(3)

In [ ]:
df_data.shape

In [ ]:
df_data.dtypes

## 2. データの中身の確認

In [ ]:
df_data.describe()

### 2.1 欠損値の有無の確認

In [ ]:
df_data.isnull().sum()

### 2.2 目的変数の表記揺れの確認

In [ ]:
df_data[y_col].value_counts()

# →表揺れなし
# 一部少数クラスあり。
# Class
# DERMASON    3546
# SIRA        2636
# SEKER       2027
# HOROZ       1928
# CALI        1630
# BARBUNYA    1322
# BOMBAY       522

### 2.3 値に0があるか確認

In [ ]:
# OpenFEを使用予定のため、0割にならないかの確認
# 0割になると、Infになり、lightgbmなどが正しく計算できなく可能性があるため。
# InfをNanに置換すれば良いが、そもそも０があるかどうかも事前に確認する
# なお、binaryカラムはないため、0はフラグとしてはではなくて、数値として計算できる


(df_data.drop(y_col, axis=1)==0).sum()

# →値に０なし

### 3. EDA

### 3.1 クラスごとに各特徴量の分布に差があるか確認

In [ ]:
dfs = []

for bean_class in df_data[y_col].unique():

    df = df_data[df_data[y_col]==bean_class].describe()
    df.columns =  [f"{bean_class}_{col}" for col in df.columns]

    dfs.append(df)
    
df_describe_by_class = pd.concat(dfs, axis=1)

In [ ]:
df_describe_by_class

In [ ]:
# # 各特徴量ごとに、各種統計量がクラスごとにばらつくかを確認する
# # ばらつきが大きければ、基本統計量において、各クラスの違いが見えやすい特徴量
# # となる可能性がある

# dfs = []
# cols = []

# for col in df_data.drop(y_col, axis=1).columns:
    
#     if col == "Area": # AreaとConvexAreaが似てるので場合分け
#         df = df_describe_by_class.filter(like=f"_{col}")
#     else:
#         df = df_describe_by_class.filter(like=col)

#     df = df.drop("count", axis=0)

#     # 特徴量ごとに桁違いなので、単位のない変動係数に変換する
#     df_cv = df.std(axis=1) / df.mean(axis=1)

#     cols.append(col)
#     dfs.append(df_cv)

# df_cvs = pd.concat(dfs, axis=1)
# df_cvs.columns = cols

In [ ]:
# df_cvs

# #        Area	Perimeter	MajorAxisLength	MinorAxisLength	AspectRatio	Eccentricity	ConvexArea	EquivDiameter	Extent	Solidity	Roundness	Compactness	ShapeFactor1	ShapeFactor2	ShapeFactor3	ShapeFactor4
# # mean	0.690888	0.319218	0.328386	0.308071	0.149174	0.115611	0.690916	0.312479	0.030381	0.002510	0.063608	0.075052	0.234999	0.405790	0.150203	0.002966
# # std	0.724484	0.373103	0.394875	0.338037	0.186727	0.432145	0.730057	0.352003	0.394247	0.331106	0.277572	0.165286	0.331660	0.368886	0.205603	0.509625
# # min	0.703016	0.317634	0.300119	0.356426	0.111511	0.301036	0.702232	0.311518	0.059700	0.017090	0.153493	0.053405	0.248474	0.339212	0.106303	0.012268
# # 25%	0.685600	0.317832	0.324826	0.307746	0.152609	0.132680	0.684185	0.311078	0.056562	0.003204	0.070000	0.076182	0.231807	0.414478	0.152922	0.003687
# # 50%	0.687469	0.320812	0.327878	0.305317	0.152925	0.115116	0.686706	0.311466	0.033955	0.002482	0.065237	0.076896	0.232614	0.412906	0.153939	0.002689
# # 75%	0.685711	0.318927	0.333042	0.304534	0.150801	0.099016	0.688141	0.310345	0.010797	0.001998	0.058753	0.076077	0.236196	0.406869	0.151778	0.001848
# # max	0.681139	0.304734	0.309091	0.306035	0.110633	0.036670	0.693053	0.317598	0.016397	0.000839	0.027157	0.055567	0.258755	0.308313	0.111077	0.000269

In [ ]:
# # 各統計量の特徴量ごとの変動係数ヒートマップで確認
# plt.figure(figsize=(14, 5))

# # seabornでヒートマップを描画
# sns.heatmap(
#     df_cvs,
#     annot=True,  
#     fmt=".4f",  # 小数点以下4桁
#     cmap="coolwarm",  
#     linewidths=0.5,  # セル間の境界線
# )

# # 見た目の調整
# plt.title("CVs Heatmap", fontsize=16, pad=15)
# plt.xticks(rotation=45, ha="right")  # X軸のラベルを斜めに
# plt.tight_layout()

# # 表示
# plt.show()

# # →各種統計量のクラスごとの変動係数ベースで
# # Areaや、CovecAreaは基本統計量全てで、クラスごとのばらつきが大きい
# # stdが、いくとかの特徴量で変動係数がほかの特徴量よりも大きいので、クラスごとに統計量にばらつきがある
# # つまり、分布が偏っているクラスもあれば、分布が広がっているクラスに分かれている可能性がある

# # 注意点：あくまで、クラスごとの特徴量ごとのばらつきなので、特徴量の重要性とイコールではないが、
# # ばらつきが大きい＝クラスごとに特徴がことなる可能性はある

In [ ]:
# 基本統計量をそのまま描画
for col in df_data.drop(y_col, axis=1).columns:
    
    if col == "Area": # AreaとConvexAreaが似てるので場合分け
        df = df_describe_by_class.filter(like=f"_{col}")
    else:
        df = df_describe_by_class.filter(like=col)

    df.drop("count", axis=0).plot.bar(colormap="tab10")

    plt.title(f"Basic statistics by class: {col}")

    plt.show()


### 3.2 差がありそうな特徴量を個別でみてみる

In [ ]:
# 一旦保留